# Automate API

In [1]:
import json
from requests import Request, Session
from requests.exceptions import ConnectionError, Timeout, TooManyRedirects

url = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"

parameters = {"start": "1", "limit": "15", "convert": "USD"}

headers = {
    "Accept": "application/json",  # Fixed typo: 'Accept' instead of 'Accepts'
    "X-CMC_PRO_API_KEY": "205db46681ca4c71801ed22e6fad4dd3",
}

session = Session()
session.headers.update(headers)

try:
    response = session.get(url, params=parameters)
    data = json.loads(response.text)

    # Uncommented so you can see the results
   # print(f"DATA => {data}")

except (ConnectionError, Timeout, TooManyRedirects) as e:
    print(e)

In [ ]:
import pandas as pd 
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

pd

# df = pd.json_normalize(data['data'])
df['Time_Stamp'] = pd.to_datetime('now')
df

In [ ]:
import json
import pandas as pd
from requests import Session


def api_fun():
    url = "https://pro-api.coinmarketcap.com/v1/cryptocurrency/listings/latest"
    parameters = {"start": "1", "limit": "15", "convert": "USD"}
    headers = {
        "Accept": "application/json",
        "X-CMC_PRO_API_KEY": "205db46681ca4c71801ed22e6fad4dd3",
    }

    session = Session()
    session.headers.update(headers)

    response = session.get(url, params=parameters)
    data = json.loads(response.text)

    df_new = pd.json_normalize(data["data"])
    df_new["Time_stamp"] = pd.Timestamp.now()

    return df_new  # Returns ONLY 15 rows


In [ ]:
from time import sleep
import os 
# Initialize an empty DataFrame OUTSIDE the loop
df_master = pd.DataFrame()

for i in range(5):
    df_new = api_fun()  # Pull 15 new rows

    # Append new 15 rows to the master DataFrame
    df_master = pd.concat([df_master, df_new], ignore_index=True)

    print(f"Run {i+1} complete. Current total rows: {len(df_master)}")
    sleep(5)
    if not  os.path.isfile(r'D:\Data_Analysis_BootCamp\Pandas\API Report.csv'):
        df_master.to_csv(r"D:\Data_Analysis_BootCamp\Pandas\API Report.csv",header='column_names')
    else:
        df_master.to_csv(r"D:\Data_Analysis_BootCamp\Pandas\API Report.csv",mode = 'a',header = False)

In [ ]:
df_master

In [ ]:
pd.set_option('display.float_format', lambda x:'%.5f' %x)

In [ ]:
df_master

In [ ]:
group = df_master.groupby("name", sort=False)[
    [
        "quote.USD.percent_change_1h",
        "quote.USD.percent_change_24h",
        "quote.USD.percent_change_7d",
        "quote.USD.percent_change_30d",
        "quote.USD.percent_change_60d",
        "quote.USD.percent_change_90d",
    ]
].mean()  # which means average
group



In [ ]:
df2 = group.stack()
df2

In [ ]:
df3 = df2.to_frame(name = 'values')
df3

In [ ]:
df3.count()

In [ ]:
index = pd.Index(range(90))
df4 = df3.reset_index()
df4

In [ ]:
df5 = df4.rename(columns = {'level_1':'Percantge_change'})
df5

In [ ]:
df6 = df5.copy()

# Map and replace the column values inside the DataFrame to find the key and replace it with value 
mapping = {
    "quote.USD.percent_change_1h": "1H",
    "quote.USD.percent_change_24h": "1D",
    "quote.USD.percent_change_7d": "7D",
    "quote.USD.percent_change_30d": "30D",
    "quote.USD.percent_change_60d": "60D",
    "quote.USD.percent_change_90d": "90D",
}

df6["Percantge_change"] = df6["Percantge_change"].replace(mapping)

In [ ]:
import seaborn as sns
import matplotlib as plt

In [ ]:
sns.catplot(x = 'Percantge_change' , y ='values' , hue = 'name',data = df6, kind = 'point')

In [ ]:
df7 =df_master[["name","quote.USD.price","Time_stamp"]]
df7 = df7.query("name == 'Bitcoin'")
df7

In [ ]:
sns.set_theme(style= 'darkgrid')
sns.lineplot(x='Time_stamp',y='quote.USD.price',hue = 'name' , data = df7)
